# Orquestador Gold — Facts

Ejecuta los 2 notebooks de tablas de hechos **en paralelo** (son independientes entre si).
**Prerequisito:** `run_gold_dims` completado exitosamente — las 5 dimensiones deben existir en `gold_ss2`.

| # | Notebook | Fuentes Silver | Target |
|---|---|---|---|
| 1 | `gold_fact_morbilidad` | 3 tablas `dbx_*` (MSPAS) | `gold_ss2.fact_morbilidad` |
| 2 | `gold_fact_defunciones` | 5 tablas INE + 2 tablas WHO | `gold_ss2.fact_defunciones` |

> **Idempotente:** todos los notebooks usan `mode('overwrite')`.
> **Timeout por notebook:** 3600 s (60 min) — las facts unen tablas de millones de filas.
> **Salida:** `'OK'` si ambos completaron, `'ERRORS'` si alguno fallo.

In [0]:
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

_ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
_this_path = _ctx.notebookPath().get()
BASE = '/'.join(_this_path.split('/')[:-2])

TIMEOUT_SEC = 3600

_NOTEBOOKS = [
    ('fact_morbilidad',   f'{BASE}/transformation-scripts/gold/gold_fact_morbilidad'),
    ('fact_defunciones',  f'{BASE}/transformation-scripts/gold/gold_fact_defunciones'),
]

print(f'Raiz del repo: {BASE}')
print(f'Notebooks a ejecutar en paralelo: {len(_NOTEBOOKS)}')

In [0]:
def run_notebook(args):
    label, path = args
    t0 = time.time()
    try:
        output = dbutils.notebook.run(path, TIMEOUT_SEC)
        return (label, 'OK', time.time() - t0, output or '')
    except Exception as e:
        return (label, f'ERROR: {e}', time.time() - t0, '')

results = []
t_global = time.time()

print('Lanzando 2 notebooks de facts en paralelo...')
with ThreadPoolExecutor(max_workers=2) as pool:
    futures = {pool.submit(run_notebook, nb): nb[0] for nb in _NOTEBOOKS}
    for future in as_completed(futures):
        label, status, elapsed, output = future.result()
        icon = 'OK' if status == 'OK' else 'ERROR'
        print(f'  [{icon}] {label:<20} {elapsed:.0f}s  {status if status != "OK" else ""}')
        if output:
            print(f'        salida: {output}')
        results.append((label, status, elapsed))

total_elapsed = time.time() - t_global

print(f'\n{"="*60}')
print('RESUMEN GOLD FACTS')
print(f'{"="*60}')
all_ok = True
for label, status, elapsed in sorted(results, key=lambda x: x[0]):
    icon = 'OK' if status == 'OK' else 'ERROR'
    print(f'  [{icon}] {label:<20} {elapsed:>6.0f}s  {"" if status == "OK" else status}')
    if status != 'OK':
        all_ok = False

print(f'\nTiempo total (paralelo): {total_elapsed:.0f}s')
print(f'Estado final: {"TODOS OK" if all_ok else "HAY ERRORES — revisa los notebooks individuales"}')
dbutils.notebook.exit('OK' if all_ok else 'ERRORS')